### Import

In [2]:
import os
import gc
import time
import pickle 
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
import matplotlib.ticker as ticker
from matplotlib import pyplot as plt 
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow.parquet import ParquetFile
from sklearn.preprocessing import OneHotEncoder
pd.set_option('display.max_columns', 500)

### General parameters

In [3]:
path_data_mimiciii = "Prediction/MIMICIII/Data/EHR/"
path_data_mimiciv  = "Prediction/MIMICIV/Data/EHR/"

path_data = "./Data/"

race_path_mimiciii = "Extraction/MIMICIII/Data/csvExtract/"
race_path_mimiciv  = "Extraction/MIMICIV/Data/csvExtract/"

### Reading Data

In [5]:
df_mimiciii = pd.read_csv(path_data_mimiciii + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_mimiciii.head(2)

In [6]:
print(df_mimiciii.ICUSTAY_ID.nunique())
print(df_mimiciii.shape)

59653
(1364554, 543)


In [7]:
df_mimiciv = pd.read_csv(path_data_mimiciv + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_mimiciv.head(2)

In [8]:
print(df_mimiciv.stay_id.nunique())
print(df_mimiciv.shape)

71344
(1638420, 526)


In [9]:
df_hirid = pd.read_csv(path_data + 'hirid_raw.csv', low_memory=False, index_col=False)
df_hirid.head(2)

In [10]:
print(df_hirid.patientid.nunique())
print(df_hirid.shape)

16642
(399055, 802)


### Drop Repeated Rows

In [11]:
all_columns = list(df_mimiciii.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]

text_columns = ['Note', 'Discharge_Note', 'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_mimiciii.drop(remove_columns, axis=1, inplace=True)
df_mimiciii = df_mimiciii.drop_duplicates()

print(df_mimiciii.ICUSTAY_ID.nunique())
print(df_mimiciii.shape)

59653
(1333367, 287)


In [12]:
all_columns = list(df_mimiciv.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]

text_columns = ['cxr_image', 'cxr_lung', 'cxr_note', 'radiology_note', 'discharge_note', 
                'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_mimiciv.drop(remove_columns, axis=1, inplace=True)
df_mimiciv = df_mimiciv.drop_duplicates()

print(df_mimiciv.stay_id.nunique())
print(df_mimiciv.shape)

71344
(1631022, 277)


### Fix Age

In [13]:
df_mimiciii.loc[df_mimiciii['AGE'] >= 95, 'AGE'] = 95
df_mimiciii = df_mimiciii[df_mimiciii.AGE > 16]

df_mimiciv.loc[df_mimiciv['age'] >= 95, 'age'] = 95
df_mimiciv = df_mimiciv[df_mimiciv.age > 16]

### Take first 24 hour LoS

In [15]:
max_rows = df_mimiciii.groupby('ICUSTAY_ID').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [16]:
max_rows = df_mimiciv.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [19]:
max_rows = df_hirid.groupby('patientid').count()
observation_window = max_rows.datetime.max()
print(observation_window)

24


In [20]:
df_mimiciii = df_mimiciii.groupby('ICUSTAY_ID').head(observation_window).reset_index(drop=True)
df_mimiciv  = df_mimiciv.groupby('stay_id').head(observation_window).reset_index(drop=True)
df_hirid    = df_hirid.groupby('patientid').head(observation_window).reset_index(drop=True)

### Find Similar Variables

In [21]:
mimiciii_columns = ['ICUSTAY_ID', 'Bins', 'AGE', 'GENDER', 'Weight', 'Height',
                    'Heart Rate', 'SpO2', 'SvO2', 'Oxygen Saturation', 'Respiratory Rate', 'Respiratory Rate (Set)',
                    'Temperature', 'Non Invasive Blood Pressure mean',
                    'Non Invasive Blood Pressure systolic', 'Non Invasive Blood Pressure diastolic',
                    'Arterial Blood Pressure mean', 'Arterial Blood Pressure systolic', 
                    'Arterial Blood Pressure diastolic',  'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                    'Bicarbonate', 'Lactate', 'Hemoglobin', 'C-Reactive Protein (CRP)',
                    'pH',  'Bilirubin, Direct', 'Bilirubin, Total', 'pO2', 'pCO2', 'PTT', 
                    'INR(PT)', 'AST', 'ALT', 'White Blood Cells', 'WBC', 'Platelet Count', 
                    'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                    'Phosphate', 'Alkaline Phosphatase', 'Mean Airway Pressure',
                    'Plateau Pressure', 'CVP', 'Amylase', 'Fibrinogen',
                    'Troponin T', 'FiO2', 'PEEP', 'PEEP (Set)', 'Total CO2',
                    'Tidal Volume', 'Tidal Volume (Set)', 'GCS Total', 'GCS - Eye Opening',  
                    'GCS - Verbal Response',  'GCS - Motor Response' , 'Richmond-RAS Scale', 'UrineOutput_IO', 
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'Antibiotic_PRC', 'Norepinephrine_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                    'Propofol_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 
                    'ICU_EXPIRE_FLAG'] 

In [22]:
mimiciv_columns = ['stay_id', 'Bins', 'age', 'gender', 'Weight', 'Height',
                   'Heart Rate', 'SpO2', 'SvO2', 'Oxygen Saturation', 'Respiratory Rate', 'Respiratory Rate (Set)',
                   'Temperature', 'Non Invasive Blood Pressure mean', 
                   'Non Invasive Blood Pressure systolic', 'Non Invasive Blood Pressure diastolic', 
                   'Arterial Blood Pressure mean', 'Arterial Blood Pressure systolic', 
                   'Arterial Blood Pressure diastolic',  'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                   'Bicarbonate',  'Lactate', 'Hemoglobin', 'C-Reactive Protein (CRP)',
                   'pH', 'Bilirubin, Direct', 'Bilirubin, Total', 'pO2', 'pCO2', 'PTT',
                   'INR(PT)', 'AST', 'ALT', 'White Blood Cells', 'WBC', 'Platelet Count', 
                   'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium', 
                   'Phosphate', 'Alkaline Phosphate', 'Mean Airway Pressure', 
                   'Plateau Pressure', 'Central Venous Pressure', 'Amylase', 'Fibrinogen',
                   'Troponin T', 'FiO2', 'PEEP',  'PEEP (Set)', 'Total CO2',
                   'Tidal Volume', 'Tidal Volume (Set)', 'Total GCS', 'GCS - Eye Opening', 
                   'GCS - Verbal Response', 'GCS - Motor Response', 'Richmond-RAS Scale', 'UrineOutput_IO', 
                   'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                   'Antibiotic_PRC', 'Norepinephrine_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                   'Propofol_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 
                   'icu_expire_flag']

In [23]:
hirid_columns = ['patientid', 'datetime', 'age',  'gender', 'weight',  'Weight', 'height',
                 'Heart Rate', 'SpO2', 'SvO2(m)', 'Oxygen Saturation_SO2', 'Respiratory Rate', 'Respiratory Rate (Set)',
                 'Temperature Central', 'Non-invasive mean arterial pressure', 
                 'Non-invasive systolic arterial pressure', 'Non-invasive diastolic arterial pressure', 
                 'Invasive mean arterial pressure', 'Invasive systolic arterial pressure',
                 'Invasive diastolic arterial pressure', 'Glucose', 'Creatinine', 'Base Excess', 'BUN',
                 'Bicarbonate', 'Lactate', 'Hemoglobin', 'C-reactive protein',
                 'pH', 'Bilirubin, Direct', 'Bilirubine, Total', 'PO2', 'pCO2', 'PTT',
                 'INR', 'AST', 'ALT', 'White Blood Cell count', 'Platelet Count',
                 'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                 'Phosphate', 'Alkaline Phosphatase', 'Mean Inspiratory Airway Pressure',
                 'Plateau Pressure', 'Central venous pressure', 'Amylase', 'Fibrinogen',
                 'Troponin-T', 'FIO2', 'Peep', 'ETCO2',
                 'TV', 'Total GCS', 'GCS Eye',
                 'GCS Verbal', 'GCS Motor', 'RASS', 'Out Urine/h',
                 'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                 'Administriation of antibiotics', 'norepinephrine', 'Heparin', 'Insulin Lantus', 'Insulin Rapid',
                 'Propofol', 'vasopressin', 'epinephrine',
                 'icu_expire_flag']

### Unit Conversion

In [24]:
conversion_factor = 18.0182
df_hirid['Glucose'] = df_hirid['Glucose'] * conversion_factor

conversion_factor = 88.4
df_hirid['Creatinine'] = df_hirid['Creatinine'] / conversion_factor

conversion_factor = 0.357
df_hirid['BUN'] = df_hirid['BUN'] / conversion_factor

conversion_factor = 10
df_hirid['Hemoglobin'] = df_hirid['Hemoglobin'] / conversion_factor

conversion_factor = 17.1
df_hirid['Bilirubin, Direct'] = df_hirid['Bilirubin, Direct'] / conversion_factor

conversion_factor = 17.1
df_hirid['Bilirubine, Total'] = df_hirid['Bilirubine, Total'] / conversion_factor

conversion_factor = 10
df_hirid['Albumin'] = df_hirid['Albumin'] / conversion_factor

conversion_factor = 100
df_hirid['Fibrinogen'] = df_hirid['Fibrinogen'] * conversion_factor

conversion_factor = 1000
df_hirid['Troponin-T'] = df_hirid['Troponin-T'] / conversion_factor

conversion_factor = 10
df_hirid['MCHC'] = df_hirid['MCHC'] / conversion_factor

In [25]:
mimiciii_columns_all = []

for i in mimiciii_columns:

    if i in list(df_mimiciii.columns):
        mimiciii_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_mimiciii.columns):
        mimiciii_columns_all.append(temp_col)

In [26]:
mimiciv_columns_all = []

for i in mimiciv_columns:

    if i in list(df_mimiciv.columns):
        mimiciv_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_mimiciv.columns):
        mimiciv_columns_all.append(temp_col)

In [27]:
hirid_columns_all = []

for i in hirid_columns:

    if i in list(df_hirid.columns):
        hirid_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_hirid.columns):
        hirid_columns_all.append(temp_col)

### Select Variables

In [28]:
df_mimiciii = df_mimiciii[mimiciii_columns_all]
df_mimiciv  = df_mimiciv[mimiciv_columns_all]
df_hirid  = df_hirid[hirid_columns_all]

In [29]:
tv_condition_iii = (df_mimiciii['Tidal Volume'].isnull()) & (df_mimiciii['Tidal Volume (Set)'].notnull())
df_mimiciii.loc[tv_condition_iii, 'Tidal Volume'] = df_mimiciii['Tidal Volume (Set)']
df_mimiciii.loc[tv_condition_iii, 'Tidal Volume_ind'] = df_mimiciii['Tidal Volume (Set)_ind']

df_mimiciii.drop(columns=['Tidal Volume (Set)', 'Tidal Volume (Set)_ind'], inplace=True)

In [30]:
tv_condition_iv = (df_mimiciv['Tidal Volume'].isnull()) & (df_mimiciv['Tidal Volume (Set)'].notnull())
df_mimiciv.loc[tv_condition_iv, 'Tidal Volume'] = df_mimiciv['Tidal Volume (Set)']
df_mimiciv.loc[tv_condition_iv, 'Tidal Volume_ind'] = df_mimiciv['Tidal Volume (Set)_ind']

df_mimiciv.drop(columns=['Tidal Volume (Set)', 'Tidal Volume (Set)_ind'], inplace=True)

In [31]:
peep_condition_iii = (df_mimiciii['PEEP'].isnull()) & (df_mimiciii['PEEP (Set)'].notnull())
df_mimiciii.loc[peep_condition_iii, 'PEEP'] = df_mimiciii['PEEP (Set)']
df_mimiciii.loc[peep_condition_iii, 'PEEP_ind'] = df_mimiciii['PEEP (Set)_ind']

df_mimiciii.drop(columns=['PEEP (Set)', 'PEEP (Set)_ind'], inplace=True)

In [32]:
peep_condition_iv = (df_mimiciv['PEEP'].isnull()) & (df_mimiciv['PEEP (Set)'].notnull())
df_mimiciv.loc[peep_condition_iv, 'PEEP'] = df_mimiciv['PEEP (Set)']
df_mimiciv.loc[peep_condition_iv, 'PEEP_ind'] = df_mimiciv['PEEP (Set)_ind']

df_mimiciv.drop(columns=['PEEP (Set)', 'PEEP (Set)_ind'], inplace=True)

In [33]:
wbc_condition_iii = (df_mimiciii['White Blood Cells'].isnull()) & (df_mimiciii['WBC'].notnull())
df_mimiciii.loc[wbc_condition_iii, 'White Blood Cells'] = df_mimiciii['WBC']
df_mimiciii.loc[wbc_condition_iii, 'White Blood Cells_ind'] = df_mimiciii['WBC_ind']

df_mimiciii.drop(columns=['WBC', 'WBC_ind'], inplace=True)

In [34]:
wbc_condition_iv = (df_mimiciv['White Blood Cells'].isnull()) & (df_mimiciv['WBC'].notnull())
df_mimiciv.loc[wbc_condition_iv, 'White Blood Cells'] = df_mimiciv['WBC']
df_mimiciv.loc[wbc_condition_iv, 'White Blood Cells_ind'] = df_mimiciv['WBC_ind']

df_mimiciv.drop(columns=['WBC', 'WBC_ind'], inplace=True)

In [35]:
weight_condition = (df_hirid['weight'].isnull()) & (df_hirid['Weight'].notnull())
df_hirid.loc[weight_condition, 'weight'] = df_hirid['Weight']

df_hirid.drop(columns=['Weight', 'Weight_ind'], inplace=True)

In [36]:
insulin_condition = (df_hirid['Insulin Rapid'].isnull()) & (df_hirid['Insulin Lantus'].notnull())
df_hirid.loc[insulin_condition, 'Insulin Rapid'] = df_hirid['Insulin Lantus']

df_hirid.drop(columns=['Insulin Lantus'], inplace=True)

In [37]:
df_hirid.drop(columns=['Administriation of antibiotics_ind', 'norepinephrine_ind', 'Heparin_ind', 
                       'Insulin Lantus_ind', 'Insulin Rapid_ind', 'Propofol_ind', 'vasopressin_ind', 
                       'epinephrine_ind'], inplace=True)

In [38]:
df_hirid.loc[df_hirid['norepinephrine'].notnull(), 'norepinephrine'] = 1
df_hirid.loc[df_hirid['Heparin'].notnull(), 'Heparin'] = 1
df_hirid.loc[df_hirid['Insulin Rapid'].notnull(), 'Insulin Rapid'] = 1
df_hirid.loc[df_hirid['Propofol'].notnull(), 'Propofol'] = 1
df_hirid.loc[df_hirid['vasopressin'].notnull(), 'vasopressin'] = 1
df_hirid.loc[df_hirid['epinephrine'].notnull(), 'epinephrine'] = 1

In [39]:
df_mimiciii.head(3)

In [40]:
df_mimiciv.head(3)

In [41]:
df_hirid.head(3)

### Unifying Column Names

In [42]:
df_mimiciii.columns = list(df_mimiciv.columns)
df_hirid.columns = list(df_mimiciv.columns)

### Outlier Detection

In [43]:
df_mimiciii.loc[df_mimiciii['Height'] < 100, 'Height'] = 100
df_mimiciv.loc[df_mimiciv['Height'] < 100, 'Height'] = 100

df_mimiciii.loc[df_mimiciii['Weight'] < 40, 'Weight'] = 40
df_mimiciv.loc[df_mimiciv['Weight'] < 40, 'Weight'] = 40

df_mimiciii.loc[df_mimiciii['Weight'] > 210, 'Weight'] = 210
df_mimiciv.loc[df_mimiciv['Weight'] > 210, 'Weight'] = 210

df_mimiciii.loc[df_mimiciii['Arterial Blood Pressure systolic'] < 10, 'Arterial Blood Pressure systolic'] = 10
df_mimiciii.loc[df_mimiciii['Arterial Blood Pressure systolic'] > 275, 'Arterial Blood Pressure systolic'] = 275
df_mimiciv.loc[df_mimiciv['Arterial Blood Pressure systolic'] > 275, 'Arterial Blood Pressure systolic'] = 275
df_hirid.loc[df_hirid['Arterial Blood Pressure systolic'] > 275, 'Arterial Blood Pressure systolic'] = 275

df_hirid.loc[df_hirid['C-Reactive Protein (CRP)'] > 350, 'C-Reactive Protein (CRP)'] = 350
df_hirid.loc[df_hirid['PTT'] > 160, 'PTT'] = 160
df_hirid.loc[df_hirid['AST'] > 1400, 'AST'] = 1400
df_hirid.loc[df_hirid['ALT'] > 1400, 'ALT'] = 1400
df_hirid.loc[df_hirid['White Blood Cells'] > 100, 'White Blood Cells'] = 100
df_hirid.loc[df_hirid['Alkaline Phosphate'] > 850, 'Alkaline Phosphate'] = 850

df_mimiciv.loc[df_mimiciv['MCHC'] > 42, 'MCHC'] = 42

df_mimiciii.loc[df_mimiciii['PaO2/FiO2'] > 40, 'PaO2/FiO2'] = 40
df_mimiciv.loc[df_mimiciv['PaO2/FiO2'] > 40, 'PaO2/FiO2'] = 40
df_hirid.loc[df_hirid['PaO2/FiO2'] > 40, 'PaO2/FiO2'] = 40

df_mimiciii.loc[df_mimiciii['Shock_Index'] < 0, 'Shock_Index'] = 0

df_mimiciii.loc[df_mimiciii['Arterial Blood Pressure diastolic'] < 0, 'Arterial Blood Pressure diastolic'] = np.nan
df_mimiciv.loc[df_mimiciv['Arterial Blood Pressure diastolic'] < 0, 'Arterial Blood Pressure diastolic'] = np.nan
df_hirid.loc[df_hirid['Arterial Blood Pressure diastolic'] < 0, 'Arterial Blood Pressure diastolic'] = np.nan

df_mimiciii.loc[df_mimiciii['Hemoglobin'] < 0, 'Hemoglobin'] = np.nan
df_mimiciv.loc[df_mimiciv['Hemoglobin'] < 0, 'Hemoglobin'] = np.nan
df_hirid.loc[df_hirid['Hemoglobin'] < 0, 'Hemoglobin'] = np.nan

df_mimiciii.loc[df_mimiciii['Bilirubin, Direct'] < 0, 'Bilirubin, Direct'] = np.nan
df_mimiciv.loc[df_mimiciv['Bilirubin, Direct'] < 0, 'Bilirubin, Direct'] = np.nan
df_hirid.loc[df_hirid['Bilirubin, Direct'] < 0, 'Bilirubin, Direct'] = np.nan

### Concatenate MIMIC Data

In [44]:
df_mimic_ = [df_mimiciii, df_mimiciv]
df_mimic = pd.concat(df_mimic_)

In [45]:
df_mimic.head()

In [46]:
df_hirid.head()

### Save Data

In [47]:
mimic_file_path = os.path.join(path_data, 'mimic.parquet')
mimic_table = pa.Table.from_pandas(df_mimic)
pq.write_table(mimic_table, mimic_file_path)

In [48]:
hirid_file_path = os.path.join(path_data, 'hirid.parquet')
hirid_table = pa.Table.from_pandas(df_hirid)
pq.write_table(hirid_table, hirid_file_path)